In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [2]:
def calculate_correlation_between_dfs(df1, df2, start_date=None, end_date=None, method='pearson', min_periods=4):
    """
    두 개의 시계열 DataFrame의 상관관계를 계산하되, 유효 관측치가 min_periods보다 많을 경우만 수행

    Parameters:
    ...
    - min_periods (int): 최소 유효 데이터 수

    Returns:
    - pd.DataFrame: 상관계수 매트릭스
    """
    if start_date:
        df1 = df1[df1.index >= pd.to_datetime(start_date)]
        df2 = df2[df2.index >= pd.to_datetime(start_date)]
    if end_date:
        df1 = df1[df1.index <= pd.to_datetime(end_date)]
        df2 = df2[df2.index <= pd.to_datetime(end_date)]

    combined = pd.merge(df1, df2, left_index=True, right_index=True, how='inner', suffixes=('_firm', '_hs'))

    corr_matrix = pd.DataFrame(index=df1.columns, columns=df2.columns, dtype=float)

    for firm in df1.columns:
        for hs in df2.columns:
            x = combined[firm]
            y = combined[hs]
            valid = x.notna() & y.notna()
            if valid.sum() >= min_periods:
                corr_matrix.loc[firm, hs] = x[valid].corr(y[valid], method=method)
            else:
                corr_matrix.loc[firm, hs] = np.nan  # 또는 0

    return corr_matrix

def get_top_correlated_hscode(corr_matrix, symbol, top_n=5, threshold=None, ascending=False):
    """
    특정 기업(Symbol)에 대해 상관관계가 높은 HS 코드를 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - symbol (str): 대상 Symbol (예: '000080')
    - top_n (int): 상위 N개 추출 (threshold와 함께 사용 시 무시될 수 있음)
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기). 설정 시 top_n보다 우선함
    - ascending (bool): 상관계수 기준 오름차순 정렬 여부 (기본값: False = 높은 값 우선)

    Returns:
    - pd.DataFrame: root_hs_code 및 상관계수를 포함한 상위 N개 HS 코드
    """

    if symbol not in corr_matrix.index:
        raise ValueError(f"Symbol '{symbol}' not found in correlation matrix.")

    symbol_corr = corr_matrix.loc[symbol].dropna()

    if threshold is not None:
        symbol_corr = symbol_corr[symbol_corr >= threshold]

    top_correlated = symbol_corr.sort_values(ascending=ascending).head(top_n)

    return top_correlated.reset_index().rename(columns={'index': 'root_hs_code', symbol: 'correlation'})

def get_top_correlated_symbols(corr_matrix, hs_code, top_n=5, threshold=None, ascending=False):
    """
    특정 HS 코드에 대해 상관관계가 높은 기업 Symbol을 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - hs_code (str or int): 대상 HS 코드 (예: '151550')
    - top_n (int): 상위 N개 추출
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기)
    - ascending (bool): 정렬 방향 (False: 높은 상관 우선)

    Returns:
    - pd.DataFrame: symbol 및 correlation 정보를 담은 상위 N개 결과
    """

    if hs_code not in corr_matrix.columns:
        raise ValueError(f"HS code '{hs_code}' not found in correlation matrix columns.")

    hs_corr = corr_matrix[hs_code].dropna()

    if threshold is not None:
        hs_corr = hs_corr[hs_corr >= threshold]

    top_symbols = hs_corr.sort_values(ascending=ascending).head(top_n)

    return top_symbols.reset_index().rename(columns={'index': 'symbol', hs_code: 'correlation'})


In [3]:
db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름
table_name = 'target_hs_code'

# 고유한 hs_code 값 추출 쿼리 실행
query = f"SELECT DISTINCT hs_code FROM {table_name}"
unique_hs_codes_df = pd.read_sql(query, con=engine)
hs_codes  = unique_hs_codes_df['hs_code'].unique().tolist()

indicator = 'expDlr'

df_real = fetch_trade_data_multi_hscode(db_info, hs_codes, indicator, 'korea_monthly_trade_data')

# 분기 정보 추가
df_real['quarter'] = df_real['date'].dt.to_period('Q')

# 그룹별로 분기별 합산
df_quarterly = (
    df_real
    .groupby(['root_hs_code', 'quarter'])['value']
    .sum()
    .reset_index()
)

# 👉 분기 월말로 변환 (예: 2007Q1 → 2007-03-31)
df_quarterly['date'] = df_quarterly['quarter'].dt.to_timestamp(how='end')

# 👉 'quarter' 컬럼 제거
df_quarterly.drop(columns=['quarter'], inplace=True)

# 1단계: 문자열로 직접 변환하려면 to_datetime 이후에 바로 strftime
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.strftime('%Y-%m-%d')

def create_yoy_growth_pivot(df_quarterly, start_date=None, end_date=None):
    """
    전년 동분기 대비 증가율을 pivot 형태로 변환하고 분석기간을 설정할 수 있는 함수

    Parameters:
    - df_quarterly (DataFrame): 'root_hs_code', 'date', 'yoy_growth' 포함된 데이터
    - start_date (str or None): 분석 시작일 (예: '2015-01-01')
    - end_date (str or None): 분석 종료일 (예: '2023-12-31')

    Returns:
    - pivot_df (DataFrame): 행: date, 열: root_hs_code, 값: yoy_growth
    """
    # Pivot
    pivot_df = df_quarterly.pivot(
        index='date',
        columns='root_hs_code',
        values='yoy_growth'
    ).sort_index()

    # inf 값 NaN 처리
    pivot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 분석 기간 슬라이싱 (날짜가 문자열이면 datetime으로 변환)
    pivot_df.index = pd.to_datetime(pivot_df.index)

    if start_date:
        pivot_df = pivot_df[pivot_df.index >= pd.to_datetime(start_date)]
    if end_date:
        pivot_df = pivot_df[pivot_df.index <= pd.to_datetime(end_date)]

    return pivot_df


# 전년 동분기 값 (4개 분기 전 값) 계산
df_quarterly['yoy_value'] = (
    df_quarterly
    .sort_values(['root_hs_code', 'date'])
    .groupby('root_hs_code')['value']
    .shift(4)
)

# ❗ yoy_growth 계산
df_quarterly['yoy_growth'] = (
    (df_quarterly['value'] - df_quarterly['yoy_value']) / df_quarterly['yoy_value']
) * 100

quarterly_trade_data = create_yoy_growth_pivot(df_quarterly, start_date='2008-03', end_date='2025-03')

In [4]:
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

KeyboardInterrupt: 

In [5]:
correlation_result = calculate_correlation_between_dfs(
    fs_yoy_growth_df,
    quarterly_trade_data,
    start_date='2020-03-31',
    end_date='2025-03-31'
)

# 상위 몇 개 확인
correlation_result.head()

root_hs_code,1201,1515500000,1515901000,1604,1703,190110,1902,190230,1902301010,19049,...,9031499090,9031809091,940130,9401309000,940340,9405,940540,9405409000,961610,961900
symbol,,,,,,,,,,,,,,,,,,,,,
A000010,-0.104806,-0.405793,0.422320,-0.448953,-0.107155,-0.322617,-0.438034,-0.355935,-0.268803,-0.212274,...,NaN,0.057465,-0.629688,-0.619018,-0.282973,0.357249,-0.834817,-0.822941,0.161411,-0.443582
A000020,-0.201809,-0.176234,0.281947,-0.681701,0.253804,0.266055,-0.138111,-0.022520,0.044538,0.420450,...,NaN,0.040714,-0.516051,-0.505482,-0.425353,0.387451,-0.587345,-0.530444,0.328273,-0.415884
A000030,-0.105759,-0.412560,0.422818,-0.458840,-0.107147,-0.304102,-0.458652,-0.374004,-0.283794,-0.213577,...,NaN,0.076597,-0.643218,-0.632563,-0.323619,0.396600,-0.837847,-0.822805,0.151904,-0.471485
A000040,0.126844,0.271092,-0.485838,0.193212,-0.236753,0.272913,-0.241248,-0.221853,-0.235290,-0.095378,...,NaN,-0.026201,-0.240429,-0.224123,-0.071190,-0.129176,0.305215,0.324601,0.011798,0.189397
A000050,0.102120,0.037008,-0.224606,0.308348,0.042973,0.096377,0.117152,0.109311,0.144210,-0.014197,...,NaN,0.462378,-0.218923,-0.224654,-0.372238,0.638642,0.244170,0.264605,-0.455046,-0.173797


#### 측정된 상관계수의 DB  업로드

In [15]:
# 인덱스를 컬럼으로 변환 (인덱스가 HS 코드인 경우)
df_reset = correlation_result.reset_index()

# 첫 번째 컬럼이 HS 코드라고 가정하고 이름을 확인
first_col = df_reset.columns[0]
print(f"\n첫 번째 컬럼명: {first_col}")

# 데이터프레임을 long format으로 변환
df_long = pd.melt(df_reset,
                  id_vars=[first_col],  # 첫 번째 컬럼을 id_vars로 사용
                  var_name='ticker',
                  value_name='value')

# hs_code 칼럼 추가 (첫 번째 컬럼 값을 사용)
df_long['hs_code'] = df_long[first_col]

# interval 칼럼 추가
df_long['interval'] = '5Y'

# 칼럼 순서 정렬
df_long = df_long[['ticker', 'hs_code', 'value', 'interval']]

# 결과 출력
print("변환된 데이터프레임:")
print(df_long.head(10))
print(f"\n전체 행 수: {len(df_long)}")

# CSV로 저장하고 싶다면:
# df_long.to_csv('correlation_data.csv', index=False)

# DB 연결 및 업로드
from sqlalchemy import create_engine
# from DATA.stock_invest_function import * # 이미 import되어 있다고 가정

# get_db_host() 함수를 사용하여 DB 정보 설정
db_info = {
    'host': get_db_host(),  # 기존 구현된 함수 사용
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

print(f"DB 연결 시도: {db_info['host']}:{db_info['port']}")

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}",
    pool_pre_ping=True,  # 연결 상태 확인
    pool_recycle=3600    # 연결 재활용 시간
)

try:
    # 연결 테스트
    with engine.connect() as conn:
        print("✓ DB 연결 성공!")

    # DataFrame을 MySQL DB에 업로드
    df_long.to_sql(
        name='correlation_table_DB',  # 테이블 이름
        con=engine,                   # DB 연결 엔진
        if_exists='replace',          # 테이블이 존재하면 대체 ('append'로 변경하면 추가)
        index=False,                  # 인덱스는 저장하지 않음
        method='multi',               # 성능 향상을 위한 bulk insert
        chunksize=10000              # 대용량 데이터를 위한 청크 처리
    )

    print("✓ 데이터가 성공적으로 DB에 업로드되었습니다!")
    print(f"테이블명: correlation_table_DB")
    print(f"업로드된 행 수: {len(df_long)}")

except Exception as e:
    print(f"❌ DB 업로드 중 오류 발생: {e}")
    print("\n연결 문제 해결 방법:")
    print("1. DB 서버가 실행 중인지 확인")
    print("2. get_db_host() 함수가 올바른 호스트 주소를 반환하는지 확인")
    print("3. 포트 번호가 정확한지 확인")
    print("4. 방화벽 설정 확인")
    print("5. DB 사용자 권한 확인")

finally:
    # 엔진 연결 해제
    engine.dispose()

# 특정 ticker와 hs_code 조합 확인
print(f"\n첫 번째 ticker(1201)의 데이터:")
print(df_long[df_long['ticker'] == '1201'].head())



첫 번째 컬럼명: symbol
변환된 데이터프레임:
  ticker  hs_code     value interval
0   1201  A000010 -0.104806       5Y
1   1201  A000020 -0.201809       5Y
2   1201  A000030 -0.105759       5Y
3   1201  A000040  0.126844       5Y
4   1201  A000050  0.102120       5Y
5   1201  A000060  0.088176       5Y
6   1201  A000070  0.136311       5Y
7   1201  A000080 -0.076823       5Y
8   1201  A000100  0.050254       5Y
9   1201  A000110  0.016549       5Y

전체 행 수: 1675542
DB 연결 시도: 192.168.0.230:3307
✓ DB 연결 성공!
✓ 데이터가 성공적으로 DB에 업로드되었습니다!
테이블명: correlation_table_DB
업로드된 행 수: 1675542

첫 번째 ticker(1201)의 데이터:
  ticker  hs_code     value interval
0   1201  A000010 -0.104806       5Y
1   1201  A000020 -0.201809       5Y
2   1201  A000030 -0.105759       5Y
3   1201  A000040  0.126844       5Y
4   1201  A000050  0.102120       5Y


#### 데이터 호출

In [5]:
# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}",
    pool_pre_ping=True,
    pool_recycle=3600
)

try:
    # 특정 interval에 해당하는 데이터만 쿼리
    query = """
    SELECT ticker, hs_code, value
    FROM correlation_table_DB
    """

    # DB에서 데이터 읽기
    df_from_db = pd.read_sql_query(query, engine)

    print(f"조회된 데이터 행 수: {len(df_from_db)}")
    print("조회된 데이터 미리보기:")
    print(df_from_db.head())

    # pivot table 생성
    correlation_pivot = df_from_db.pivot_table(
        index='ticker',      # 행: ticker
        columns='hs_code',   # 열: hs_code
        values='value',      # 값: correlation value
        aggfunc='first'      # 중복값이 있을 경우 첫 번째 값 사용
    )

    print(f"\nPivot table 생성 완료:")
    print(f"Shape: {correlation_pivot.shape}")
    print(f"Index (ticker) 수: {len(correlation_pivot.index)}")
    print(f"Columns (hs_code) 수: {len(correlation_pivot.columns)}")

    # 변수명 생성 (고정된 이름 사용)
    var_name = 'correlation'
    print(f"\n생성될 변수명: {var_name}")

    # 전역 변수로 할당
    globals()[var_name] = correlation_pivot

    print(f"\n✓ '{var_name}' 데이터프레임이 생성되었습니다!")

    # 데이터 미리보기
    print(f"\n데이터프레임 미리보기:")
    print(correlation_pivot.head())

    # 인덱스와 컬럼 정보 확인
    print(f"\n인덱스 (ticker) 샘플:")
    print(correlation_pivot.index[:5].tolist())

    print(f"\n컬럼 (hs_code) 샘플:")
    print(correlation_pivot.columns[:5].tolist())

    # 특정 셀 값 확인
    if len(correlation_pivot) > 0 and len(correlation_pivot.columns) > 0:
        first_ticker = correlation_pivot.index[0]
        first_hs_code = correlation_pivot.columns[0]
        sample_value = correlation_pivot.loc[first_ticker, first_hs_code]
        print(f"\n샘플 상관계수 값:")
        print(f"ticker '{first_ticker}' - hs_code '{first_hs_code}': {sample_value}")

except Exception as e:
    print(f"❌ 오류 발생: {e}")
    print("오류 상세 정보:")
    import traceback
    traceback.print_exc()


조회된 데이터 행 수: 1675542
조회된 데이터 미리보기:
  ticker  hs_code     value
0   1201  A000010 -0.104806
1   1201  A000020 -0.201809
2   1201  A000030 -0.105759
3   1201  A000040  0.126844
4   1201  A000050  0.102120

Pivot table 생성 완료:
Shape: (458, 2504)
Index (ticker) 수: 458
Columns (hs_code) 수: 2504

생성될 변수명: correlation

✓ 'correlation' 데이터프레임이 생성되었습니다!

데이터프레임 미리보기:
hs_code      A000010   A000020   A000030   A000040   A000050   A000060  \
ticker                                                                   
1201       -0.104806 -0.201809 -0.105759  0.126844  0.102120  0.088176   
1515500000 -0.405793 -0.176234 -0.412560  0.271092  0.037008  0.234528   
1515901000  0.422320  0.281947  0.422818 -0.485838 -0.224606 -0.378232   
1604       -0.448953 -0.681701 -0.458840  0.193212  0.308348  0.095283   
1703       -0.107155  0.253804 -0.107147 -0.236753  0.042973  0.026298   

hs_code      A000070   A000080   A000100   A000110  ...   A900310   A900340  \
ticker                                    

In [6]:
correlation_pivot.T

ticker,1201,1515500000,1515901000,1604,1703,190110,1902,190230,1902301010,19049,...,9031499000,9031809091,940130,9401309000,940340,9405,940540,9405409000,961610,961900
hs_code,,,,,,,,,,,,,,,,,,,,,
A000010,-0.104806,-0.405793,0.422320,-0.448953,-0.107155,-0.322617,-0.438034,-0.355935,-0.268803,-0.212274,...,0.755148,0.057465,-0.629688,-0.619018,-0.282973,0.357249,-0.834817,-0.822941,0.161411,-0.443582
A000020,-0.201809,-0.176234,0.281947,-0.681701,0.253804,0.266055,-0.138111,-0.022520,0.044538,0.420450,...,0.093487,0.040714,-0.516051,-0.505482,-0.425353,0.387451,-0.587345,-0.530444,0.328273,-0.415884
A000030,-0.105759,-0.412560,0.422818,-0.458840,-0.107147,-0.304102,-0.458652,-0.374004,-0.283794,-0.213577,...,0.745446,0.076597,-0.643218,-0.632563,-0.323619,0.396600,-0.837847,-0.822805,0.151904,-0.471485
A000040,0.126844,0.271092,-0.485838,0.193212,-0.236753,0.272913,-0.241248,-0.221853,-0.235290,-0.095378,...,0.059274,-0.026201,-0.240429,-0.224123,-0.071190,-0.129176,0.305215,0.324601,0.011798,0.189397
A000050,0.102120,0.037008,-0.224606,0.308348,0.042973,0.096377,0.117152,0.109311,0.144210,-0.014197,...,-0.403766,0.462378,-0.218923,-0.224654,-0.372238,0.638642,0.244170,0.264605,-0.455046,-0.173797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A950160,0.013983,0.059054,0.362741,-0.494389,0.157447,-0.018179,-0.268788,-0.207050,-0.230939,-0.004918,...,0.390960,-0.426845,0.563987,0.590662,-0.065291,-0.129401,-0.403714,-0.389337,0.219009,-0.168678
A950170,0.028558,-0.551562,0.267629,-0.293799,0.034905,-0.325381,-0.338085,-0.287469,-0.234937,-0.187475,...,0.675154,0.161883,-0.166988,-0.153736,-0.231197,0.155227,-0.558212,-0.532007,0.249512,-0.243240
A950180,-0.001366,-0.544369,0.441496,-0.388739,0.029914,-0.147233,-0.568258,-0.565651,-0.556841,-0.196214,...,0.366699,0.002050,-0.221289,-0.212928,-0.395689,0.100867,-0.424662,-0.394315,0.168912,-0.486620


In [12]:
top_symbols = get_top_correlated_symbols(
    corr_matrix = correlation_pivot.T,
    hs_code= '850422',
    top_n= 50,
    threshold=0.1  # 선택사항
)
print(top_symbols)

    hs_code  correlation
0   A032620     0.791796
1   A010620     0.783249
2   A103230     0.739828
3   A050890     0.730159
4   A031820     0.723238
5   A264900     0.719523
6   A054780     0.717314
7   A044490     0.706842
8   A050090     0.704819
9   A051500     0.703187
10  A099410     0.699289
11  A033600     0.684596
12  A900140     0.680375
13  A003090     0.678668
14  A023910     0.674428
15  A020560     0.673201
16  A009540     0.670607
17  A036000     0.662652
18  A010120     0.660999
19  A031990     0.659939
20  A052690     0.659673
21  A153710     0.653196
22  A000650     0.651144
23  A001040     0.649783
24  A298040     0.648665
25  A097780     0.645046
26  A007980     0.643691
27  A006730     0.640712
28  A006250     0.640265
29  A084670     0.637120
30  A024890     0.634922
31  A063570     0.631491
32  A084680     0.629873
33  A267790     0.628723
34  A263750     0.624853
35  A112240     0.623857
36  A011760     0.623253
37  A001540     0.619702
38  A000020     0.614908


In [15]:
top_hs_codes = get_top_correlated_hscode(
    corr_matrix=correlation_pivot.T,  # 이전에 만든 상관관계 행렬
    symbol ='A298040',
    top_n=100,
    threshold=0.3  # 선택사항
)

print(top_hs_codes.head(50))

        ticker  correlation
0   8529109210     0.842089
1   3402120000     0.735006
2         9405     0.654211
3       850422     0.648665
4   3912909000     0.620487
5   9026102000     0.608904
6   2930904090     0.577034
7       850423     0.568200
8   3208909019     0.561441
9   8517704090     0.558404
10  8517702000     0.544361
11        8481     0.543981
12  2930903010     0.538338
13      853720     0.527805
14  8501534000     0.508960
15      481910     0.506516
16  8472901090     0.505033
17      850433     0.502655
18  8506500000     0.498581
19      850650     0.498581
20  8504501090     0.489937
21  8479899099     0.477875
22        9028     0.460723
23  8905909000     0.457602
24  8455301000     0.447580
25  2922422000     0.431496
26      730791     0.419178
27  7307910000     0.419178
28      845931     0.418523
29  8207401000     0.417060
30  8455309000     0.411735
31      847170     0.396962
32        5515     0.385881
33  4002709000     0.376794
34      847989     0

In [11]:
# from pykrx import stock
#
# # 1. ticker 리스트 불러오기
# tickers = stock.get_market_ticker_list(market="ALL")
#
# # 2. ticker와 name을 리스트로 만들기
# data = []
# for t in tickers:
#     name = stock.get_market_ticker_name(t)
#     data.append({
#         'ticker': t,
#         'name': name,
#         'symbol': 'A' + t
#     })
#
# # 3. DataFrame으로 변환
# company_name_df = pd.DataFrame(data, columns=['symbol', 'ticker', 'name'])


In [10]:
correlation_result.loc['A298040'][['850422']]

NameError: name 'correlation_result' is not defined

In [ ]:
pd.merge(top_symbols, company_name_df, on='symbol', how = 'left')

In [15]:
hscode = fetch_table_data(db_info, "target_hs_code")

hscode[hscode['hs_code'] == '0']

✅ 'target_hs_code' 테이블에서 766건의 데이터를 가져왔습니다.


,hs_code
